In [ ]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null # Instalar SDK java 17

In [ ]:
!wget -q https://downloads.apache.org/spark/spark-4.0.2/spark-4.0.2-bin-hadoop3.tgz # Descargar Spark 4.0.1

In [ ]:
!tar xf spark-4.0.2-bin-hadoop3.tgz # Descomprimir la version de Spark

In [ ]:
!pip install -q findspark # Instalar la librería findspark

In [ ]:
!pip install -q pyspark # Instalar pyspark

# Funciones de agregación

Son las mismas que SQL, buscan mediante una operación matemática extraer una característica de un conjunto de datos.

Los **niveles de agrupación** que admite Spark son:

- Tratar a un **dataframe** como un grupo.
- Dividir un **dataframe** en varios grupos usando una o más columnas y realizar agregaciones en cada una de ellas.
- Divdir un **dataframe** en varias ventanas y realizar una media móvil, suma acumulativa o una clasificación.

Todas las agregaciones se realizan a través de funciones.


In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
df = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/vuelos.parquet')

In [ ]:
df.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [ ]:
df.show(20, truncate=False)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

## Función COUNT, COUNT DISTINCT y APPROX_COUNT_DISTINCT

In [ ]:
from pyspark.sql.functions import count

df.select(
    count('*').alias('Registros totales')
).show()

+-----------------+
|Registros totales|
+-----------------+
|          5819079|
+-----------------+



In [ ]:
# CountDistinct
from pyspark.sql.functions import countDistinct

df.select(
    countDistinct('ORIGIN_AIRPORT').alias('Origen distintos')
).show()

+----------------+
|Origen distintos|
+----------------+
|             628|
+----------------+



In [ ]:
# aprox_count_distinct: da una aproximacion

from pyspark.sql.functions import approx_count_distinct

df.select(
    approx_count_distinct('ORIGIN_AIRPORT', 0.3).alias('Origen distintos')
).show()


+----------------+
|Origen distintos|
+----------------+
|             802|
+----------------+



## Funciones MIN y MAX

In [ ]:
from pyspark.sql.functions import min, max

df.select(
    min('DEPARTURE_DELAY').alias('Minimo retraso'),
    max('DEPARTURE_DELAY').alias('Maximo retraso')
).show()

+--------------+--------------+
|Minimo retraso|Maximo retraso|
+--------------+--------------+
|           -82|          1988|
+--------------+--------------+



## Funciones SUM SUMDISTINCT y AVG

In [ ]:
from pyspark.sql.functions import sum, sum_distinct, avg, round

df.select(
    sum('DEPARTURE_DELAY').alias('Retraso total'),
    sum_distinct('DEPARTURE_DELAY').alias('Retraso total sin duplicados'),
    round(avg('DEPARTURE_TIME'), 3).alias('Promedio de tiempo de salida redondeado')).show()

+-------------+----------------------------+---------------------------------------+
|Retraso total|Retraso total sin duplicados|Promedio de tiempo de salida redondeado|
+-------------+----------------------------+---------------------------------------+
|     53718424|                      710072|                               1335.204|
+-------------+----------------------------+---------------------------------------+



## Agregación con agrupación

A veces dados como los datos están mostrados es conveniente aplicar las funciones de agregación a grupos dados por una determinada condición indicada en nuestro código, tal como pasa con las sentencias SQL.

Pondré un solo ejempo con count, pero puede haber infinitos casos más con todas las funciones vistas previamente.

In [ ]:
from pyspark.sql.functions import desc

In [ ]:
(df.groupby('ORIGIN_AIRPORT')
.count()
.orderBy(desc('count'))
.show())

+--------------+------+
|ORIGIN_AIRPORT| count|
+--------------+------+
|           ATL|346836|
|           ORD|285884|
|           DFW|239551|
|           DEN|196055|
|           LAX|194673|
|           SFO|148008|
|           PHX|146815|
|           IAH|146622|
|           LAS|133181|
|           MSP|112117|
|           MCO|110982|
|           SEA|110899|
|           DTW|108500|
|           BOS|107847|
|           EWR|101772|
|           CLT|100324|
|           LGA| 99605|
|           SLC| 97210|
|           JFK| 93811|
|           BWI| 86079|
+--------------+------+
only showing top 20 rows


## Pivot

Convierte los valores únicos de una columna específica en nuevas columnas separadas, no haré ejemplos con este set de datos dado que es horrible como ejemplo didactico.

## Joins

Los dataframes son tratados como conjuntos y junto a estos la teoría de conjuntos es totalmente aplicable.

Si tenemos dos conjuntos o más podremos usar los diferentes tipos de JOIN al igual que en SQL.

In [48]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [51]:
empleados = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/empleados.parquet')
departamentos = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/departamentos.parquet')

In [52]:
empleados.show()

+------+--------+
|nombre|num_dpto|
+------+--------+
|  Luis|      33|
| Katia|      33|
|  Raul|      34|
| Pedro|       0|
| Laura|      34|
|Sandro|      31|
+------+--------+



In [53]:
departamentos.show()

+---+-----------+
| id|nombre_dpto|
+---+-----------+
| 31|     letras|
| 33|    derecho|
| 34| matemática|
| 35|informática|
+---+-----------+



In [54]:
from pyspark.sql.functions import col

### Inner Join

In [58]:
join_df = empleados.join(departamentos, col('num_dpto') == col('id'))
join_df.show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+



In [60]:
# Podremos también realizar el JOIN comparando los campos y listo, al igual que en las grandes consultas SQL

join_df2 = empleados.join(departamentos, empleados.num_dpto == departamentos.id)
join_df2.show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+



### Left Outer Join

In [61]:
empleados.join(departamentos, empleados.num_dpto == departamentos.id, 'left').show()

+------+--------+----+-----------+
|nombre|num_dpto|  id|nombre_dpto|
+------+--------+----+-----------+
|  Luis|      33|  33|    derecho|
| Katia|      33|  33|    derecho|
|  Raul|      34|  34| matemática|
| Pedro|       0|NULL|       NULL|
| Laura|      34|  34| matemática|
|Sandro|      31|  31|     letras|
+------+--------+----+-----------+



### Rigth Outer Join

In [62]:
empleados.join(departamentos, empleados.num_dpto == departamentos.id, 'right').show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|Sandro|      31| 31|     letras|
| Katia|      33| 33|    derecho|
|  Luis|      33| 33|    derecho|
| Laura|      34| 34| matemática|
|  Raul|      34| 34| matemática|
|  NULL|    NULL| 35|informática|
+------+--------+---+-----------+



### Full Outer Join

In [63]:
empleados.join(departamentos, empleados.num_dpto == departamentos.id, 'full').show()

+------+--------+----+-----------+
|nombre|num_dpto|  id|nombre_dpto|
+------+--------+----+-----------+
| Pedro|       0|NULL|       NULL|
|Sandro|      31|  31|     letras|
|  Luis|      33|  33|    derecho|
| Katia|      33|  33|    derecho|
|  Raul|      34|  34| matemática|
| Laura|      34|  34| matemática|
|  NULL|    NULL|  35|informática|
+------+--------+----+-----------+



### Left Anti Join

In [64]:
empleados.join(departamentos, empleados.num_dpto == departamentos.id, 'left_anti').show()

+------+--------+
|nombre|num_dpto|
+------+--------+
| Pedro|       0|
+------+--------+



### Left Semi Join

In [65]:
empleados.join(departamentos, empleados.num_dpto == departamentos.id, 'left_semi').show()

+------+--------+
|nombre|num_dpto|
+------+--------+
|  Luis|      33|
| Katia|      33|
|  Raul|      34|
| Laura|      34|
|Sandro|      31|
+------+--------+



### Cross Join

In [66]:
empleados.crossJoin(departamentos).show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 31|     letras|
|  Luis|      33| 33|    derecho|
|  Luis|      33| 34| matemática|
|  Luis|      33| 35|informática|
| Katia|      33| 31|     letras|
| Katia|      33| 33|    derecho|
| Katia|      33| 34| matemática|
| Katia|      33| 35|informática|
|  Raul|      34| 31|     letras|
|  Raul|      34| 33|    derecho|
|  Raul|      34| 34| matemática|
|  Raul|      34| 35|informática|
| Pedro|       0| 31|     letras|
| Pedro|       0| 33|    derecho|
| Pedro|       0| 34| matemática|
| Pedro|       0| 35|informática|
| Laura|      34| 31|     letras|
| Laura|      34| 33|    derecho|
| Laura|      34| 34| matemática|
| Laura|      34| 35|informática|
+------+--------+---+-----------+
only showing top 20 rows


### Shuffle Hash Join

In [67]:
from pyspark.sql.functions import col, broadcast

In [68]:
empleados.join(broadcast(departamentos), col('num_dpto') == col('id')).show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+



### Broadcast Hash Join

In [69]:
empleados.join(broadcast(departamentos), col('num_dpto') == col('id')).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [num_dpto#1146L], [id#1147L], Inner, BuildRight, false
   :- Filter isnotnull(num_dpto#1146L)
   :  +- FileScan parquet [nombre#1145,num_dpto#1146L] Batched: true, DataFilters: [isnotnull(num_dpto#1146L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/drive/MyDrive/RDD_FILES/sparksql/empleados.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(num_dpto)], ReadSchema: struct<nombre:string,num_dpto:bigint>
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=2015]
      +- Filter isnotnull(id#1147L)
         +- FileScan parquet [id#1147L,nombre_dpto#1148] Batched: true, DataFilters: [isnotnull(id#1147L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/drive/MyDrive/RDD_FILES/sparksql/departamentos.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(id)], ReadSchema: struct<id:bigint,nombre_dpto:string>


